# Vectorization and Modeling

### Vectorization (TF-idF)

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

print("1. Loading the master dataset...")
df = pd.read_csv('../data/interim/soft_balanced_final.csv')

# Combine headline and body for the text vectorizers
df['combined_text'] = df['headline_clean'].fillna('') + " " + df['body_clean'].fillna('')

# Define our features and target
X_text = df['combined_text']
y = df['stance']

# Define the numerical features we engineered
num_cols = [
    'headline_sentiment', 'body_sentiment', 'sentiment_gap',
    'headline_length_words', 'body_length_words',
    'overlap_clean', 'headline_coverage_clean', 'body_coverage_clean',
    'ner_overlap_count', 'ner_overlap_ratio'
]
X_num = df[num_cols].fillna(0) # Fill any accidental NaNs with 0


1. Loading the master dataset...


In [2]:
print("2. Performing strict Train/Val/Test splits...")
# First split: Separate out the 15% Test set
text_trainval, text_test, num_trainval, num_test, y_trainval, y_test = train_test_split(
    X_text, X_num, y, test_size=0.15, stratify=y, random_state=42
)

# Second split: Separate the remaining 85% into Train and Validation
text_train, text_val, num_train, num_val, y_train, y_val = train_test_split(
    text_trainval, num_trainval, y_trainval, test_size=0.1765, stratify=y_trainval, random_state=42
)

print("3. Scaling numerical features...")
# Tree models don't strictly need scaling, but it is best practice (and required if you ever run Logistic Regression again)
scaler = StandardScaler()
num_train_scaled = scaler.fit_transform(num_train)
num_val_scaled = scaler.transform(num_val)
num_test_scaled = scaler.transform(num_test)

2. Performing strict Train/Val/Test splits...
3. Scaling numerical features...


In [3]:
print("4. Processing Route A: TF-IDF + TruncatedSVD...")
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 3), sublinear_tf=True)
tfidf_train = tfidf.fit_transform(text_train)
tfidf_val = tfidf.transform(text_val)
tfidf_test = tfidf.transform(text_test)

svd = TruncatedSVD(n_components=300, random_state=42)
svd_train = svd.fit_transform(tfidf_train)
svd_val = svd.transform(tfidf_val)
svd_test = svd.transform(tfidf_test)

4. Processing Route A: TF-IDF + TruncatedSVD...


### Dense Embedding

In [4]:
print("5. Processing Route B: Dense Embeddings (This will take a few minutes)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embed_train = embedder.encode(text_train.tolist(), show_progress_bar=True)
embed_val = embedder.encode(text_val.tolist(), show_progress_bar=True)
embed_test = embedder.encode(text_test.tolist(), show_progress_bar=True)

5. Processing Route B: Dense Embeddings (This will take a few minutes)...


Batches:   0%|          | 0/1033 [00:00<?, ?it/s]

Batches:   0%|          | 0/222 [00:00<?, ?it/s]

Batches:   0%|          | 0/222 [00:00<?, ?it/s]

In [7]:
print("\n6. Saving all matrices to disk...")
os.makedirs('../data/vectors', exist_ok=True)

# Save Labels
np.save('../data/vectors/y_train.npy', y_train)
np.save('../data/vectors/y_val.npy', y_val)
np.save('../data/vectors/y_test.npy', y_test)

# Save Scaled Numerical Features
np.save('../data/vectors/num_train.npy', num_train_scaled)
np.save('../data/vectors/num_val.npy', num_val_scaled)
np.save('../data/vectors/num_test.npy', num_test_scaled)

# Save SVD Text Vectors
np.save('../data/vectors/svd_train.npy', svd_train)
np.save('../data/vectors/svd_val.npy', svd_val)
np.save('../data/vectors/svd_test.npy', svd_test)

# Save Embedding Text Vectors
np.save('../data/vectors/embed_train.npy', embed_train)
np.save('../data/vectors/embed_val.npy', embed_val)
np.save('../data/vectors/embed_test.npy', embed_test)

print("=== Vectorization Complete! ===")


6. Saving all matrices to disk...
=== Vectorization Complete! ===


### Modeling

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report
import time

def run_baseline_suite(dataset_name, X_train, y_train, X_val, y_val):
    print(f"--- Running Suite on: {dataset_name} ---")
    
    # Define our baseline competitors
    # Note: We use GaussianNB because SVD and Embeddings create dense matrices with negative values.
    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced'),
        "Gaussian Naive Bayes": GaussianNB(),
        "Random Forest (Baseline)": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
    }
    
    results = []
    
    for name, model in models.items():
        print(f"Training {name}...")
        start_time = time.time()
        
        # Train
        model.fit(X_train, y_train)
        
        # Predict
        y_pred = model.predict(X_val)
        
        # Evaluate
        macro_f1 = f1_score(y_val, y_pred, average='macro')
        train_time = round(time.time() - start_time, 2)
        
        results.append({
            "Dataset": dataset_name,
            "Model": name,
            "Validation Macro F1": round(macro_f1, 4),
            "Train Time (sec)": train_time
        })
        
    print("Suite Complete!\n")
    return pd.DataFrame(results)

In [12]:
# 1. Load the Labels
y_train = np.load('../data/vectors/y_train.npy', allow_pickle=True)
y_val = np.load('../data/vectors/y_val.npy', allow_pickle=True)

# 2. Load Route A (TF-IDF + SVD)
X_train_svd = np.load('../data/vectors/svd_train.npy')
X_val_svd = np.load('../data/vectors/svd_val.npy')

# 3. Load Route B (Dense Embeddings)
X_train_embed = np.load('../data/vectors/embed_train.npy')
X_val_embed = np.load('../data/vectors/embed_val.npy')

# 4. Run the experiments!
results_svd = run_baseline_suite("Route A (TF-IDF + SVD)", X_train_svd, y_train, X_val_svd, y_val)
results_embed = run_baseline_suite("Route B (Dense Embeddings)", X_train_embed, y_train, X_val_embed, y_val)

# 5. Combine and view the master leaderboard
master_leaderboard = pd.concat([results_svd, results_embed], ignore_index=True)
master_leaderboard = master_leaderboard.sort_values(by="Validation Macro F1", ascending=False)
print(master_leaderboard)

--- Running Suite on: Route A (TF-IDF + SVD) ---
Training Logistic Regression...
Training Gaussian Naive Bayes...
Training Random Forest (Baseline)...
Suite Complete!

--- Running Suite on: Route B (Dense Embeddings) ---
Training Logistic Regression...
Training Gaussian Naive Bayes...
Training Random Forest (Baseline)...
Suite Complete!

                      Dataset                     Model  Validation Macro F1  \
5  Route B (Dense Embeddings)  Random Forest (Baseline)               0.7560   
2      Route A (TF-IDF + SVD)  Random Forest (Baseline)               0.6886   
3  Route B (Dense Embeddings)       Logistic Regression               0.6576   
0      Route A (TF-IDF + SVD)       Logistic Regression               0.5788   
4  Route B (Dense Embeddings)      Gaussian Naive Bayes               0.4892   
1      Route A (TF-IDF + SVD)      Gaussian Naive Bayes               0.4449   

   Train Time (sec)  
5             47.11  
2             43.52  
3              2.22  
0          

In [13]:
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

# The 9 Diverse Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, 
    ExtraTreesClassifier, 
    AdaBoostClassifier, 
    HistGradientBoostingClassifier # Faster version of LightGBM/XGBoost built into sklearn
)

# 1. Load Data (assuming you saved them from Notebook 4)
y_train = np.load('../data/vectors/y_train.npy', allow_pickle=True)
y_val = np.load('../data/vectors/y_val.npy', allow_pickle=True)

X_train_svd = np.load('../data/vectors/svd_train.npy')
X_val_svd = np.load('../data/vectors/svd_val.npy')

X_train_embed = np.load('../data/vectors/embed_train.npy')
X_val_embed = np.load('../data/vectors/embed_val.npy')


In [14]:
# 2. Define the 9 Pipelines
# We use StandardScaler in the pipeline because models like SVC, KNN, and LR mathematically require scaled features.
pipelines = {
    "Logistic Regression": Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))]),
    "Gaussian NB": Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]),
    "Linear SVC": Pipeline([('scaler', StandardScaler()), ('clf', LinearSVC(class_weight='balanced', max_iter=2000, random_state=42))]),
    "KNN": Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))]),
    "Decision Tree": Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(class_weight='balanced', random_state=42))]),
    "Random Forest": Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=15, random_state=42))]),
    "Extra Trees": Pipeline([('scaler', StandardScaler()), ('clf', ExtraTreesClassifier(n_estimators=100, class_weight='balanced', max_depth=15, random_state=42))]),
    "AdaBoost": Pipeline([('scaler', StandardScaler()), ('clf', AdaBoostClassifier(random_state=42))]),
    "HistGradientBoosting": Pipeline([('scaler', StandardScaler()), ('clf', HistGradientBoostingClassifier(class_weight='balanced', random_state=42))])
}


In [15]:
# 3. Define the Master Evaluation Function
def evaluate_pipelines(dataset_name, X_train, y_train, X_val, y_val, pipelines):
    print(f"\nEvaluating {dataset_name}...")
    results = []
    
    for name, pipeline in pipelines.items():
        print(f"  Training {name}...")
        start_time = time.time()
        
        # Train the pipeline
        pipeline.fit(X_train, y_train)
        train_time = round(time.time() - start_time, 2)
        
        # Predict on Train (to check for overfitting)
        y_train_pred = pipeline.predict(X_train)
        train_macro_f1 = f1_score(y_train, y_train_pred, average='macro')
        
        # Predict on Val (Actual Performance)
        y_val_pred = pipeline.predict(X_val)
        val_macro_f1 = f1_score(y_val, y_val_pred, average='macro')
        
        # Calculate Overfit Penalty (Train F1 - Val F1)
        overfit_gap = train_macro_f1 - val_macro_f1
        
        # Get detailed class-wise metrics as a dictionary
        report = classification_report(y_val, y_val_pred, output_dict=True, zero_division=0)
        
        # Construct the detailed row
        row = {
            "Dataset": dataset_name,
            "Model": name,
            "Train Time (s)": train_time,
            "Train Macro F1": round(train_macro_f1, 4),
            "Val Macro F1": round(val_macro_f1, 4),
            "Overfit Gap": round(overfit_gap, 4),
            
            # Class: Agree
            "Agree Precision": round(report['agree']['precision'], 4),
            "Agree Recall": round(report['agree']['recall'], 4),
            "Agree F1": round(report['agree']['f1-score'], 4),
            
            # Class: Disagree
            "Disagree Precision": round(report['disagree']['precision'], 4),
            "Disagree Recall": round(report['disagree']['recall'], 4),
            "Disagree F1": round(report['disagree']['f1-score'], 4),
            
            # Class: Discuss
            "Discuss Precision": round(report['discuss']['precision'], 4),
            "Discuss Recall": round(report['discuss']['recall'], 4),
            "Discuss F1": round(report['discuss']['f1-score'], 4),
            
            # Class: Unrelated
            "Unrelated Precision": round(report['unrelated']['precision'], 4),
            "Unrelated Recall": round(report['unrelated']['recall'], 4),
            "Unrelated F1": round(report['unrelated']['f1-score'], 4)
        }
        
        results.append(row)
        
    return pd.DataFrame(results)


In [16]:
# 4. Execute the Suite for both Data Routes
df_svd = evaluate_pipelines("TF-IDF + SVD", X_train_svd, y_train, X_val_svd, y_val, pipelines)
df_embed = evaluate_pipelines("Dense Embeddings", X_train_embed, y_train, X_val_embed, y_val, pipelines)



Evaluating TF-IDF + SVD...
  Training Logistic Regression...
  Training Gaussian NB...
  Training Linear SVC...
  Training KNN...
  Training Decision Tree...
  Training Random Forest...
  Training Extra Trees...
  Training AdaBoost...


d:\Anaconda\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


  Training HistGradientBoosting...

Evaluating Dense Embeddings...
  Training Logistic Regression...
  Training Gaussian NB...
  Training Linear SVC...
  Training KNN...
  Training Decision Tree...
  Training Random Forest...
  Training Extra Trees...
  Training AdaBoost...


d:\Anaconda\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


  Training HistGradientBoosting...


In [17]:
# 5. Combine, Sort, and Save
master_results = pd.concat([df_svd, df_embed], ignore_index=True)

# Sort by the highest Validation F1 Score
master_results = master_results.sort_values(by="Val Macro F1", ascending=False).reset_index(drop=True)

print("\n=== Top 5 Models ===")
print(master_results[['Dataset', 'Model', 'Train Macro F1', 'Val Macro F1', 'Overfit Gap']].head())

# Save to CSV for your documentation
master_results.to_csv("../data/models_evaluation_report.csv", index=False)
print("\nFull detailed report saved to 'data/models_evaluation_report.csv'")


=== Top 5 Models ===
            Dataset                 Model  Train Macro F1  Val Macro F1  \
0  Dense Embeddings         Random Forest          0.9858        0.8559   
1  Dense Embeddings           Extra Trees          0.9817        0.8520   
2  Dense Embeddings  HistGradientBoosting          0.9436        0.8437   
3      TF-IDF + SVD         Random Forest          0.9415        0.7893   
4      TF-IDF + SVD  HistGradientBoosting          0.8785        0.7736   

   Overfit Gap  
0       0.1298  
1       0.1297  
2       0.0999  
3       0.1522  
4       0.1049  

Full detailed report saved to 'data/models_evaluation_report.csv'
